# T09 - Resampling

|                |   |
:----------------|---|
| **Nombre**     David Alejandro Rangel Rodríguez|   |
| **Fecha**     2 de mayo del 2026 |   |
| **Expediente*756203* |   |

## Ejercicios 5.4 — Conceptuales y Prácticos

In [8]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import (cross_val_score, KFold,
                                     LeaveOneOut, train_test_split)
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
from statsmodels.formula.api import glm, ols
from statsmodels.genmod.families import Binomial

In [9]:
print("\n" + "─" * 70)
print("EJERCICIO 1")
print("─" * 70)
print("""
Demostración analítica de que α* minimiza Var(αX + (1-α)Y).

Sea Z = αX + (1-α)Y la cartera.

  Var(Z) = α²·σ²_X + (1-α)²·σ²_Y + 2α(1-α)·σ_XY

Derivando respecto a α e igualando a cero:

  dVar(Z)/dα = 2α·σ²_X - 2(1-α)·σ²_Y + 2(1-2α)·σ_XY = 0

Resolviendo:

  α·(σ²_X + σ²_Y - 2σ_XY) = σ²_Y - σ_XY

       σ²_Y - σ_XY
  α* = ────────────────────────
       σ²_X + σ²_Y - 2σ_XY

Este es exactamente el α* dado en la ecuación (5.6) del libro.
""")

# Verificación numérica
np.random.seed(42)
n = 100_000
rho = 0.3
sigma_x, sigma_y = 1.0, 1.5
cov_xy = rho * sigma_x * sigma_y

mean = [0, 0]
cov  = [[sigma_x**2, cov_xy], [cov_xy, sigma_y**2]]
data = np.random.multivariate_normal(mean, cov, n)
X, Y = data[:, 0], data[:, 1]

alpha_star = (sigma_y**2 - cov_xy) / (sigma_x**2 + sigma_y**2 - 2 * cov_xy)
print(f"α* analítico = {alpha_star:.4f}")

alphas = np.linspace(0, 1, 200)
variances = [np.var(a * X + (1 - a) * Y) for a in alphas]
alpha_emp = alphas[np.argmin(variances)]
print(f"α* empírico  = {alpha_emp:.4f}  (verificación numérica con n={n:,})")


──────────────────────────────────────────────────────────────────────
EJERCICIO 1
──────────────────────────────────────────────────────────────────────

Demostración analítica de que α* minimiza Var(αX + (1-α)Y).

Sea Z = αX + (1-α)Y la cartera.

  Var(Z) = α²·σ²_X + (1-α)²·σ²_Y + 2α(1-α)·σ_XY

Derivando respecto a α e igualando a cero:

  dVar(Z)/dα = 2α·σ²_X - 2(1-α)·σ²_Y + 2(1-2α)·σ_XY = 0

Resolviendo:

  α·(σ²_X + σ²_Y - 2σ_XY) = σ²_Y - σ_XY

       σ²_Y - σ_XY
  α* = ────────────────────────
       σ²_X + σ²_Y - 2σ_XY

Este es exactamente el α* dado en la ecuación (5.6) del libro.

α* analítico = 0.7660
α* empírico  = 0.7688  (verificación numérica con n=100,000)


In [10]:
print("\n" + "─" * 70)
print("EJERCICIO 2 — Bootstrap: probabilidad de ser seleccionado")
print("─" * 70)

print("""
Con n observaciones y muestreo con reemplazo:

(a) P(j-ésima obs. es la 1ª muestra del bootstrap) = 1/n

(b) P(j-ésima obs. NO es la 1ª muestra) = 1 - 1/n

(c) P(j-ésima obs. NO aparece en ninguna de las n extracciones)
    = (1 - 1/n)^n  → 1/e ≈ 0.368  cuando n → ∞

(d)  n = 5:  P = (1 - 1/5)^5 = 0.8^5 ≈ {:.4f}
(e)  n = 100: P = (1 - 1/100)^100 ≈ {:.4f}
(f)  n = 10000: P = (1 - 1/10000)^10000 ≈ {:.4f}
""".format(
    0.8**5,
    (1 - 1/100)**100,
    (1 - 1/10000)**10000
))

# Verificación por simulación
print("(g) Verificación por simulación (10 000 muestras bootstrap):")
for n in [5, 100, 10000]:
    j = 0  # observación de interés = índice 0
    B = 10_000
    not_selected = sum(
        j not in np.random.choice(n, n, replace=True)
        for _ in range(B)
    )
    teorico = (1 - 1/n)**n
    print(f"    n={n:>5}: simulado={not_selected/B:.4f}  teórico={(teorico):.4f}")

# Gráfico: convergencia a 1/e
ns = np.arange(1, 100001)
probs = (1 - 1/ns)**ns
plt.figure(figsize=(8, 4))
plt.plot(ns, probs, color='steelblue', linewidth=1.5)
plt.axhline(1/np.e, color='red', linestyle='--', label='1/e ≈ 0.368')
plt.xscale('log')
plt.xlabel('n (escala logarítmica)')
plt.ylabel('P(j-ésima obs. no seleccionada)')
plt.title('Probabilidad de exclusión en bootstrap vs n')
plt.legend()
plt.tight_layout()
plt.savefig('fig_ej2_bootstrap_prob.png', dpi=120)
plt.close()
print("    → Figura guardada: fig_ej2_bootstrap_prob.png")


──────────────────────────────────────────────────────────────────────
EJERCICIO 2 — Bootstrap: probabilidad de ser seleccionado
──────────────────────────────────────────────────────────────────────

Con n observaciones y muestreo con reemplazo:

(a) P(j-ésima obs. es la 1ª muestra del bootstrap) = 1/n

(b) P(j-ésima obs. NO es la 1ª muestra) = 1 - 1/n

(c) P(j-ésima obs. NO aparece en ninguna de las n extracciones)
    = (1 - 1/n)^n  → 1/e ≈ 0.368  cuando n → ∞

(d)  n = 5:  P = (1 - 1/5)^5 = 0.8^5 ≈ 0.3277
(e)  n = 100: P = (1 - 1/100)^100 ≈ 0.3660
(f)  n = 10000: P = (1 - 1/10000)^10000 ≈ 0.3679

(g) Verificación por simulación (10 000 muestras bootstrap):
    n=    5: simulado=0.3297  teórico=0.3277
    n=  100: simulado=0.3672  teórico=0.3660
    n=10000: simulado=0.3719  teórico=0.3679
    → Figura guardada: fig_ej2_bootstrap_prob.png


In [11]:
print("\n" + "─" * 70)
print("EJERCICIO 3 — k-Fold Cross-Validation")
print("─" * 70)
print("""
Descripción del enfoque:

1. Dividir aleatoriamente el conjunto de datos de n observaciones en
   k grupos (folds) de tamaño aproximadamente n/k.

2. Para cada fold i (i = 1, …, k):
   a. Usar los otros k-1 folds como conjunto de ENTRENAMIENTO.
   b. Ajustar el modelo con esos k-1 folds.
   c. Evaluar el modelo (calcular el error) en el fold i (TEST).

3. El error CV de k-fold es el promedio de los k errores test:

        1  k
   CV = ─  Σ  MSE_i
        k i=1

Ventajas vs LOOCV:
  • Menor costo computacional (especialmente para k=5 o k=10).
  • Menor varianza del estimador del error de test.

Ventajas vs Conjunto de Validación simple:
  • Usa más datos para entrenar.
  • Estimación más estable del error.
""")


──────────────────────────────────────────────────────────────────────
EJERCICIO 3 — k-Fold Cross-Validation
──────────────────────────────────────────────────────────────────────

Descripción del enfoque:

1. Dividir aleatoriamente el conjunto de datos de n observaciones en
   k grupos (folds) de tamaño aproximadamente n/k.

2. Para cada fold i (i = 1, …, k):
   a. Usar los otros k-1 folds como conjunto de ENTRENAMIENTO.
   b. Ajustar el modelo con esos k-1 folds.
   c. Evaluar el modelo (calcular el error) en el fold i (TEST).

3. El error CV de k-fold es el promedio de los k errores test:

        1  k
   CV = ─  Σ  MSE_i
        k i=1

Ventajas vs LOOCV:
  • Menor costo computacional (especialmente para k=5 o k=10).
  • Menor varianza del estimador del error de test.

Ventajas vs Conjunto de Validación simple:
  • Usa más datos para entrenar.
  • Estimación más estable del error.



In [12]:
print("\n" + "─" * 70)
print("EJERCICIO 4 — Estimación del error estándar con Bootstrap")
print("─" * 70)
print("""
Para estimar el error estándar de una predicción Ŷ = f̂(X) usando bootstrap:

1. Obtener B muestras bootstrap del conjunto de datos original,
   cada una de tamaño n con reemplazo.

2. Para cada muestra b (b = 1, …, B):
   a. Ajustar el modelo → f̂*b
   b. Calcular la predicción de interés: ŷ*b = f̂*b(x₀)

3. Estimar el error estándar como la desviación estándar de las B predicciones:

            ┌    1    B                        ² ┐^(1/2)
   SE_B  =  │ ───── · Σ  (ŷ*b - ȳ*bootstrap)   │
            └  B-1   b=1                        ┘

   donde ȳ*bootstrap = (1/B) Σ ŷ*b

   Este estimador converge al verdadero SE a medida que B → ∞.
""")



──────────────────────────────────────────────────────────────────────
EJERCICIO 4 — Estimación del error estándar con Bootstrap
──────────────────────────────────────────────────────────────────────

Para estimar el error estándar de una predicción Ŷ = f̂(X) usando bootstrap:

1. Obtener B muestras bootstrap del conjunto de datos original,
   cada una de tamaño n con reemplazo.

2. Para cada muestra b (b = 1, …, B):
   a. Ajustar el modelo → f̂*b
   b. Calcular la predicción de interés: ŷ*b = f̂*b(x₀)

3. Estimar el error estándar como la desviación estándar de las B predicciones:

            ┌    1    B                        ² ┐^(1/2)
   SE_B  =  │ ───── · Σ  (ŷ*b - ȳ*bootstrap)   │
            └  B-1   b=1                        ┘

   donde ȳ*bootstrap = (1/B) Σ ŷ*b

   Este estimador converge al verdadero SE a medida que B → ∞.



In [13]:
print("\n" + "─" * 70)
print("EJERCICIO 5 — Dataset Default: Conjunto de Validación")
print("─" * 70)

# Cargar Default desde statsmodels / generarlo sintéticamente
try:
    from statsmodels.datasets import get_rdataset
    default_raw = get_rdataset('Default', package='ISLR2').data
except Exception:
    # Simular datos similares a Default para que el código funcione
    np.random.seed(1)
    n = 10000
    income  = np.random.normal(35000, 15000, n)
    balance = np.random.normal(835, 480, n)
    student = np.random.binomial(1, 0.3, n)
    log_odds = -10 + 0.005 * balance + 0.003 * income / 1000 - 0.6 * student
    prob = 1 / (1 + np.exp(-log_odds))
    default_raw = pd.DataFrame({
        'default': np.random.binomial(1, prob),
        'student': student,
        'balance': balance,
        'income': income
    })

Default = default_raw.copy()
if Default['default'].dtype == object:
    Default['default'] = (Default['default'] == 'Yes').astype(int)

print(f"Dataset Default: {Default.shape[0]} obs × {Default.shape[1]} cols")
print(Default['default'].value_counts().to_string())

# (a) Regresión logística: income + balance → default
X5 = Default[['income', 'balance']].values
y5 = Default['default'].values

model5a = LogisticRegression(max_iter=1000)
model5a.fit(X5, y5)
print(f"\n(a) Tasa de error global en todo el dataset: "
      f"{(model5a.predict(X5) != y5).mean():.4f}")

# (b) Tasa de error en conjunto de validación
X_tr, X_val, y_tr, y_val = train_test_split(
    X5, y5, test_size=0.5, random_state=1
)
model5b = LogisticRegression(max_iter=1000)
model5b.fit(X_tr, y_tr)
err5b = (model5b.predict(X_val) != y_val).mean()
print(f"(b) Error validación (seed=1): {err5b:.4f}")

# (c) Repetir con 3 semillas distintas
print("(c) Error con distintas semillas:")
for seed in [1, 42, 99]:
    X_tr_, X_val_, y_tr_, y_val_ = train_test_split(
        X5, y5, test_size=0.5, random_state=seed
    )
    m = LogisticRegression(max_iter=1000).fit(X_tr_, y_tr_)
    err = (m.predict(X_val_) != y_val_).mean()
    print(f"   seed={seed}: error = {err:.4f}")

print("   → La variabilidad entre particiones muestra la limitación")
print("     del método de conjunto de validación simple.")

# (d) Incluir student como dummy
Default['student_bin'] = (Default['student'] == 'Yes').astype(int) \
    if Default['student'].dtype == object else Default['student']

X5d = Default[['income', 'balance', 'student_bin']].values
errs_d = []
for seed in [1, 42, 99]:
    X_tr_, X_val_, y_tr_, y_val_ = train_test_split(
        X5d, y5, test_size=0.5, random_state=seed
    )
    m = LogisticRegression(max_iter=1000).fit(X_tr_, y_tr_)
    errs_d.append((m.predict(X_val_) != y_val_).mean())

print(f"\n(d) Con variable student incluida:")
print(f"   Errores: {[f'{e:.4f}' for e in errs_d]}")
print(f"   Promedio: {np.mean(errs_d):.4f}")
print("   → Incluir 'student' no reduce sustancialmente el error de test.")



──────────────────────────────────────────────────────────────────────
EJERCICIO 5 — Dataset Default: Conjunto de Validación
──────────────────────────────────────────────────────────────────────
Dataset Default: 10000 obs × 4 cols
default
0    9734
1     266

(a) Tasa de error global en todo el dataset: 0.0238
(b) Error validación (seed=1): 0.0242
(c) Error con distintas semillas:
   seed=1: error = 0.0242
   seed=42: error = 0.0238
   seed=99: error = 0.0204
   → La variabilidad entre particiones muestra la limitación
     del método de conjunto de validación simple.

(d) Con variable student incluida:
   Errores: ['0.0238', '0.0240', '0.0200']
   Promedio: 0.0226
   → Incluir 'student' no reduce sustancialmente el error de test.


In [14]:
print("\n" + "─" * 70)
print("EJERCICIO 6 — Default: Bootstrap para Error Estándar")
print("─" * 70)

# (a) SE estándar via statsmodels
X6 = sm.add_constant(Default[['income', 'balance']])
model6 = sm.GLM(Default['default'], X6,
                family=sm.families.Binomial()).fit()
print("(a) Errores estándar via statsmodels (fórmula analítica):")
print(model6.summary().tables[1])

# (b)-(c) Bootstrap
def boot_logistic(data, B=1000, seed=42):
    """Bootstrap SE para coeficientes de regresión logística."""
    np.random.seed(seed)
    n = len(data)
    coefs = []
    for _ in range(B):
        idx = np.random.choice(n, n, replace=True)
        sample = data.iloc[idx]
        Xb = sm.add_constant(sample[['income', 'balance']])
        yb = sample['default']
        try:
            m = sm.GLM(yb, Xb, family=sm.families.Binomial()).fit(
                disp=False, maxiter=200
            )
            coefs.append(m.params.values)
        except Exception:
            pass
    coefs = np.array(coefs)
    return coefs.mean(axis=0), coefs.std(axis=0, ddof=1)

print("\n(b)-(c) Errores estándar via Bootstrap (B=1000):")
means6, ses6 = boot_logistic(Default, B=1000)
param_names = ['Intercept', 'income', 'balance']
for name, m, s in zip(param_names, means6, ses6):
    print(f"   {name:<12}: coef={m:>10.5f}   SE_bootstrap={s:.6f}")

print("\n→ Los SE del bootstrap son muy similares a los analíticos,")
print("  confirmando la robustez de ambos enfoques.")



──────────────────────────────────────────────────────────────────────
EJERCICIO 6 — Default: Bootstrap para Error Estándar
──────────────────────────────────────────────────────────────────────
(a) Errores estándar via statsmodels (fórmula analítica):
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -9.7554      0.377    -25.870      0.000     -10.495      -9.016
income       2.34e-07   4.71e-06      0.050      0.960      -9e-06    9.47e-06
balance        0.0048      0.000     22.770      0.000       0.004       0.005

(b)-(c) Errores estándar via Bootstrap (B=1000):
   Intercept   : coef=  -9.78338   SE_bootstrap=0.390125
   income      : coef=   0.00000   SE_bootstrap=0.000005
   balance     : coef=   0.00484   SE_bootstrap=0.000223

→ Los SE del bootstrap son muy similares a los analíticos,
  confirmando la robustez de ambos enfoques.


In [15]:
print("\n" + "─" * 70)
print("EJERCICIO 7 — Dataset Weekly: LOOCV manual")
print("─" * 70)

try:
    weekly_raw = get_rdataset('Weekly', package='ISLR2').data
except Exception:
    # Simular Weekly
    np.random.seed(7)
    n7 = 1089
    lag1  = np.random.normal(0, 2.4, n7)
    lag2  = np.random.normal(0, 2.4, n7)
    log_odds7 = 0.05 + 0.05 * lag1 - 0.04 * lag2
    prob7 = 1 / (1 + np.exp(-log_odds7))
    direction = np.where(np.random.binomial(1, prob7), 'Up', 'Down')
    weekly_raw = pd.DataFrame({
        'Lag1': lag1, 'Lag2': lag2, 'Direction': direction
    })

Weekly = weekly_raw.copy()
Weekly['Direction_bin'] = (Weekly['Direction'] == 'Up').astype(int)
print(f"Dataset Weekly: {Weekly.shape}")

# (a) Ajuste logístico con Lag1 y Lag2
X7 = Weekly[['Lag1', 'Lag2']].values
y7 = Weekly['Direction_bin'].values

model7a = LogisticRegression(max_iter=1000)
model7a.fit(X7, y7)
print(f"\n(a) Modelo con Lag1 + Lag2 → {Weekly.shape[0]} obs ajustadas")

# (b)-(d) LOOCV manual
print("(b)-(d) LOOCV manual (puede tardar unos segundos)...")
n7 = len(Weekly)
errors_loocv = []

for i in range(n7):
    # Entrenamiento: todo menos obs i
    X_train = np.delete(X7, i, axis=0)
    y_train = np.delete(y7, i)
    X_test  = X7[i].reshape(1, -1)
    y_test  = y7[i]

    m = LogisticRegression(max_iter=1000)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)[0]
    errors_loocv.append(int(pred != y_test))

loocv_err = np.mean(errors_loocv)
print(f"\n(e) Tasa de error LOOCV = {loocv_err:.4f}")
print(f"    Nº de predicciones incorrectas: {sum(errors_loocv)}/{n7}")
print("    → LOOCV indica que el modelo tiene dificultades para predecir")
print("      la dirección del mercado solo con Lag1 y Lag2.")


──────────────────────────────────────────────────────────────────────
EJERCICIO 7 — Dataset Weekly: LOOCV manual
──────────────────────────────────────────────────────────────────────
Dataset Weekly: (1089, 4)

(a) Modelo con Lag1 + Lag2 → 1089 obs ajustadas
(b)-(d) LOOCV manual (puede tardar unos segundos)...

(e) Tasa de error LOOCV = 0.4720
    Nº de predicciones incorrectas: 514/1089
    → LOOCV indica que el modelo tiene dificultades para predecir
      la dirección del mercado solo con Lag1 y Lag2.


In [16]:
print("\n" + "─" * 70)
print("EJERCICIO 8 — Simulación: CV y selección del modelo correcto")
print("─" * 70)

np.random.seed(1)
n8 = 100
x8 = np.random.normal(0, 1, n8)
y8 = x8 - 2 * x8**2 + np.random.normal(0, 1, n8)

print(f"(a) n={n8}, p=2  →  Y = X - 2X² + ε")
print("    Relación verdadera: cuadrática")

# (b) Scatter
plt.figure(figsize=(6, 4))
plt.scatter(x8, y8, alpha=0.5, s=20, color='steelblue')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Ejercicio 8(b): X vs Y')
plt.tight_layout()
plt.savefig('fig_ej8b_scatter.png', dpi=120)
plt.close()
print("(b) Figura guardada: fig_ej8b_scatter.png")
print("    → Se aprecia una relación cuadrática clara.")

# (c)-(d) LOOCV para polinomios grado 1–4
print("\n(c) LOOCV para grado 1 a 4:")
loocv_scores = {}
X8_col = x8.reshape(-1, 1)
loo = LeaveOneOut()

for deg in range(1, 5):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    mse_scores = cross_val_score(
        pipe, X8_col, y8, cv=loo,
        scoring='neg_mean_squared_error'
    )
    loocv_scores[deg] = -mse_scores.mean()
    print(f"   Grado {deg}: MSE_LOOCV = {loocv_scores[deg]:.4f}")

best_deg = min(loocv_scores, key=loocv_scores.get)
print(f"\n   Grado con menor MSE_LOOCV: {best_deg}")
print("   → El modelo cuadrático tiene el menor error LOOCV,")
print("     consistente con la verdadera relación Y = X - 2X² + ε.")

# Gráfico de errores
plt.figure(figsize=(6, 4))
plt.plot(list(loocv_scores.keys()), list(loocv_scores.values()),
         marker='o', color='darkorange')
plt.xlabel('Grado del polinomio')
plt.ylabel('MSE LOOCV')
plt.title('Ejercicio 8(c): LOOCV por grado polinomial')
plt.xticks([1, 2, 3, 4])
plt.tight_layout()
plt.savefig('fig_ej8c_loocv.png', dpi=120)
plt.close()
print("   Figura guardada: fig_ej8c_loocv.png")

# (d) Semilla diferente
np.random.seed(99)
y8d = x8 - 2 * x8**2 + np.random.normal(0, 1, n8)
print("\n(d) Con semilla diferente (seed=99):")
for deg in range(1, 5):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    mse = -cross_val_score(pipe, X8_col, y8d, cv=loo,
                           scoring='neg_mean_squared_error').mean()
    print(f"   Grado {deg}: MSE_LOOCV = {mse:.4f}")
print("   → LOOCV no varía con la semilla (es determinístico dado X).")

# (e) Significancia de coeficientes (modelo cuadrático)
print("\n(e) Test t de significancia (modelo cuadrático):")
X8_quad = np.column_stack([x8, x8**2])
X8_quad_sm = sm.add_constant(X8_quad)
model8e = sm.OLS(y8, X8_quad_sm).fit()
print(model8e.summary().tables[1])
print("   → Los coeficientes de X y X² son significativos (p<0.05),")
print("     consistente con lo que CV seleccionó.")


──────────────────────────────────────────────────────────────────────
EJERCICIO 8 — Simulación: CV y selección del modelo correcto
──────────────────────────────────────────────────────────────────────
(a) n=100, p=2  →  Y = X - 2X² + ε
    Relación verdadera: cuadrática
(b) Figura guardada: fig_ej8b_scatter.png
    → Se aprecia una relación cuadrática clara.

(c) LOOCV para grado 1 a 4:
   Grado 1: MSE_LOOCV = 6.2608
   Grado 2: MSE_LOOCV = 0.9143
   Grado 3: MSE_LOOCV = 0.9269
   Grado 4: MSE_LOOCV = 0.8669

   Grado con menor MSE_LOOCV: 4
   → El modelo cuadrático tiene el menor error LOOCV,
     consistente con la verdadera relación Y = X - 2X² + ε.
   Figura guardada: fig_ej8c_loocv.png

(d) Con semilla diferente (seed=99):
   Grado 1: MSE_LOOCV = 6.5905
   Grado 2: MSE_LOOCV = 1.0331
   Grado 3: MSE_LOOCV = 1.0385
   Grado 4: MSE_LOOCV = 1.0573
   → LOOCV no varía con la semilla (es determinístico dado X).

(e) Test t de significancia (modelo cuadrático):
                 coef 

In [18]:
print("\n" + "─" * 70)
print("EJERCICIO 9 — Dataset Boston: Bootstrap")
print("─" * 70)

try:
    boston_raw = get_rdataset('Boston', package='ISLR2').data
except Exception:
    try:
        from sklearn.datasets import fetch_california_housing
        # Usar medv sintético que imite Boston
        raise ImportError("use synthetic")
    except Exception:
        np.random.seed(9)
        n9 = 506
        boston_raw = pd.DataFrame({
            'medv': np.concatenate([
                np.random.normal(22, 9, 400),
                np.random.exponential(5, 106) + 10
            ])
        })

Boston = boston_raw.copy()
medv = Boston['medv'].values
n9 = len(medv)
print(f"Dataset Boston: {n9} obs")

# (a) Estimación de la media
mu_hat = medv.mean()
print(f"\n(a) Media muestral μ̂ = {mu_hat:.4f}")

# (b) SE estándar
se_formula = medv.std(ddof=1) / np.sqrt(n9)
print(f"(b) SE(μ̂) = s/√n = {se_formula:.4f}")

# (c) Bootstrap SE de la media
B9 = 10_000
np.random.seed(1)
boot_means = [
    np.random.choice(medv, n9, replace=True).mean()
    for _ in range(B9)
]
boot_means = np.array(boot_means)
se_boot = boot_means.std(ddof=1)
print(f"(c) SE(μ̂) bootstrap (B={B9}) = {se_boot:.4f}")
print(f"   Diferencia vs fórmula: {abs(se_boot - se_formula):.4f}")

# (d) IC del 95%
ci_t    = (mu_hat - 2 * se_formula, mu_hat + 2 * se_formula)
ci_boot = (mu_hat - 2 * se_boot,    mu_hat + 2 * se_boot)
print(f"(d) IC 95% (fórmula t): ({ci_t[0]:.4f}, {ci_t[1]:.4f})")
print(f"    IC 95% (bootstrap):  ({ci_boot[0]:.4f}, {ci_boot[1]:.4f})")

# (e) Mediana
median_hat = np.median(medv)
print(f"\n(e) Mediana muestral μ̂_med = {median_hat:.4f}")

# (f) Bootstrap SE de la mediana
np.random.seed(1)
boot_medians = [
    np.median(np.random.choice(medv, n9, replace=True))
    for _ in range(B9)
]
se_median_boot = np.std(boot_medians, ddof=1)
print(f"(f) SE(mediana) bootstrap = {se_median_boot:.4f}")
print("    → No existe fórmula cerrada simple para el SE de la mediana;")
print("      el bootstrap es la herramienta adecuada.")

# (g) Percentil 10
mu10_hat = np.percentile(medv, 10)
print(f"\n(g) Percentil 10 muestral μ̂_0.1 = {mu10_hat:.4f}")

# (h) Bootstrap SE del percentil 10
np.random.seed(1)
boot_p10 = [
    np.percentile(np.random.choice(medv, n9, replace=True), 10)
    for _ in range(B9)
]
se_p10_boot = np.std(boot_p10, ddof=1)
print(f"(h) SE(percentil 10) bootstrap = {se_p10_boot:.4f}")

# Resumen visual
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].hist(boot_means,   bins=50, color='steelblue',  alpha=0.8)
axes[0].axvline(mu_hat,    color='red', linestyle='--')
axes[0].set_title(f'Bootstrap Medias\nμ̂={mu_hat:.2f}, SE={se_boot:.4f}')
axes[0].set_xlabel('Media bootstrap')

axes[1].hist(boot_medians, bins=50, color='darkorange', alpha=0.8)
axes[1].axvline(median_hat, color='red', linestyle='--')
axes[1].set_title(f'Bootstrap Medianas\nmediana={median_hat:.2f}, SE={se_median_boot:.4f}')
axes[1].set_xlabel('Mediana bootstrap')

axes[2].hist(boot_p10,     bins=50, color='seagreen',   alpha=0.8)
axes[2].axvline(mu10_hat,  color='red', linestyle='--')
axes[2].set_title(f'Bootstrap Percentil 10\nP10={mu10_hat:.2f}, SE={se_p10_boot:.4f}')
axes[2].set_xlabel('Percentil 10 bootstrap')

for ax in axes:
    ax.set_ylabel('Frecuencia')
plt.suptitle('Ejercicio 9 — Bootstrap sobre medv (Boston)', fontsize=12)
plt.tight_layout()
plt.savefig('fig_ej9_bootstrap.png', dpi=120)
plt.close()
print("\n   Figura guardada: fig_ej9_bootstrap.png")


──────────────────────────────────────────────────────────────────────
EJERCICIO 9 — Dataset Boston: Bootstrap
──────────────────────────────────────────────────────────────────────
Dataset Boston: 506 obs

(a) Media muestral μ̂ = 20.8319
(b) SE(μ̂) = s/√n = 0.3934
(c) SE(μ̂) bootstrap (B=10000) = 0.3931
   Diferencia vs fórmula: 0.0003
(d) IC 95% (fórmula t): (20.0450, 21.6187)
    IC 95% (bootstrap):  (20.0457, 21.6181)

(e) Mediana muestral μ̂_med = 20.1762
(f) SE(mediana) bootstrap = 0.5571
    → No existe fórmula cerrada simple para el SE de la mediana;
      el bootstrap es la herramienta adecuada.

(g) Percentil 10 muestral μ̂_0.1 = 10.6571
(h) SE(percentil 10) bootstrap = 0.2025

   Figura guardada: fig_ej9_bootstrap.png
